<a href="https://colab.research.google.com/github/Kenny625819/Applied-Data-Science/blob/main/%E3%82%84%E3%82%8A%E7%9B%B4%E3%81%97%E3%81%9F%E3%81%A3%E3%81%9F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
ESCC (Bilsky scale) Inter-Rater Agreement and Consensus Label Generation
[177-case version]

This script performs the following steps:

1. Load ESCC ratings from an Excel file (three raters per case).
2. Compute overall inter-rater agreement using Fleiss' kappa.
3. Compute pairwise Cohen's kappa (unweighted and quadratic-weighted)
   for each rater pair.
4. Generate a consensus ESCC label for each case:
   - If at least two raters agree, the majority label is used.
   - If all three raters disagree, the median label is chosen based on
     the ordinal Bilsky scale.
5. Save the original ratings and the consensus label to a new Excel file.

Expected input format
---------------------
The input Excel file must contain the following columns:

    ESCC_1  : ESCC label from rater 1 (e.g., "1a", "1b", "1c", "2", "3")
    ESCC_2  : ESCC label from rater 2
    ESCC_3  : ESCC label from rater 3

The Bilsky/ESCC labels are treated as an ordinal scale with the following order:
    1a < 1b < 1c < 2 < 3
"""

import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics import cohen_kappa_score


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
INPUT_EXCEL = "ESCC_3_177.xlsx"
OUTPUT_EXCEL = "ESCC_3_with_consensus_177.xlsx"

# Bilsky/ESCC ordinal scale
ORDERED_LABELS = ["1a", "1b", "1c", "2", "3"]

LABEL_TO_IDX = {lab: i for i, lab in enumerate(ORDERED_LABELS)}
IDX_TO_LABEL = {i: lab for lab, i in LABEL_TO_IDX.items()}


# ---------------------------------------------------------------------
# Fleiss' kappa
# ---------------------------------------------------------------------
def fleiss_kappa(M: np.ndarray):
    """
    Compute Fleiss' kappa for multiple raters.

    Parameters
    ----------
    M : np.ndarray of shape (N, k)
        M[i, j] is the number of raters who assigned category j to case i.

    Returns
    -------
    kappa : float
        Fleiss' kappa.
    P_bar : float
        Mean observed agreement.
    P_e : float
        Expected agreement by chance.
    """
    N, k = M.shape
    n = M.sum(axis=1)[0]  # number of raters per case (assumed constant)

    p_j = M.sum(axis=0) / (N * n)
    P_i = ((M * (M - 1)).sum(axis=1)) / (n * (n - 1))

    P_bar = P_i.mean()
    P_e = (p_j ** 2).sum()

    kappa = (P_bar - P_e) / (1 - P_e)
    return kappa, P_bar, P_e


# ---------------------------------------------------------------------
# Quadratic-weighted Cohen's kappa
# ---------------------------------------------------------------------
def quadratic_weighted_kappa(a, b, labels_order):
    """
    Compute quadratic-weighted Cohen's kappa between two raters.
    """
    lab_to_i = {lab: i for i, lab in enumerate(labels_order)}
    a_idx = np.array([lab_to_i[str(x).strip()] for x in a])
    b_idx = np.array([lab_to_i[str(x).strip()] for x in b])
    n_cat = len(labels_order)

    # Weight matrix
    w = np.zeros((n_cat, n_cat))
    for i in range(n_cat):
        for j in range(n_cat):
            w[i, j] = ((i - j) ** 2) / ((n_cat - 1) ** 2)

    # Observed matrix
    O = np.zeros((n_cat, n_cat))
    for x, y in zip(a_idx, b_idx):
        O[x, y] += 1
    O = O / O.sum()

    # Expected matrix
    a_hist = np.bincount(a_idx, minlength=n_cat)
    b_hist = np.bincount(b_idx, minlength=n_cat)
    E = np.outer(a_hist, b_hist)
    E = E / E.sum()

    num = (w * O).sum()
    den = (w * E).sum()
    return 1.0 - num / den


# ---------------------------------------------------------------------
# Consensus label generation
# ---------------------------------------------------------------------
def consensus_row(row, raters, label_to_idx, idx_to_label):
    """
    Generate a consensus ESCC label for a single case.

    Rules:
    - If at least two raters agree, return the majority label.
    - If all three raters disagree, return the median label on the
      ordinal Bilsky scale (1a < 1b < 1c < 2 < 3).
    """
    vals = [str(row[x]).strip() for x in raters]
    counts = Counter(vals)
    most_common_label, freq = counts.most_common(1)[0]

    if freq >= 2:
        return most_common_label

    idxs = sorted(label_to_idx[v] for v in vals)
    median_idx = idxs[1]
    return idx_to_label[median_idx]


# ---------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------
def main():
    print(f"Loading input file: {INPUT_EXCEL}")
    df = pd.read_excel(INPUT_EXCEL)
    print("Input columns:", list(df.columns))
    print(f"Number of cases: {len(df)}")

    raters = ["ESCC_1", "ESCC_2", "ESCC_3"]

    # Check required columns
    missing_cols = [c for c in raters if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    # Standardize labels as strings
    for c in raters:
        df[c] = df[c].astype(str).str.strip()

    # Build N x k matrix for Fleiss' kappa
    N = len(df)
    k = len(ORDERED_LABELS)
    M = np.zeros((N, k), dtype=int)

    for i, row in df[raters].iterrows():
        for val in row:
            if pd.isna(val):
                continue
            if val not in LABEL_TO_IDX:
                raise ValueError(f"Unexpected ESCC label found: {val}")
            M[i, LABEL_TO_IDX[val]] += 1

    # Fleiss' kappa
    fleiss_k, P_bar, P_e = fleiss_kappa(M)
    print("\n=== Fleiss' kappa ===")
    print(f"Fleiss' kappa: {fleiss_k:.3f}")
    print(f"P_bar (mean agreement): {P_bar:.3f}")
    print(f"P_e (chance agreement): {P_e:.3f}")

    # Pairwise Cohen's kappa
    print("\n=== Pairwise Cohen's kappa ===")
    pairs = [("ESCC_1", "ESCC_2"), ("ESCC_1", "ESCC_3"), ("ESCC_2", "ESCC_3")]

    for c1, c2 in pairs:
        a = df[c1].values
        b = df[c2].values

        k_unw = cohen_kappa_score(a, b)
        k_w = quadratic_weighted_kappa(a, b, ORDERED_LABELS)

        print(f"{c1} vs {c2}: Unweighted = {k_unw:.3f}, Quadratic-weighted = {k_w:.3f}")

    # Consensus labels
    df["ESCC_consensus"] = df.apply(
        consensus_row,
        axis=1,
        args=(raters, LABEL_TO_IDX, IDX_TO_LABEL),
    )

    print("\n=== Consensus ESCC distribution ===")
    print(df["ESCC_consensus"].value_counts().sort_index())

    full_disagree = (
        (df["ESCC_1"] != df["ESCC_2"]) &
        (df["ESCC_1"] != df["ESCC_3"]) &
        (df["ESCC_2"] != df["ESCC_3"])
    ).sum()

    print(f"\nNumber of full-disagreement cases: {full_disagree}")

    # Save output
    df.to_excel(OUTPUT_EXCEL, index=False)
    print(f"\nSaved consensus results to: {OUTPUT_EXCEL}")


if __name__ == "__main__":
    main()


Loading input file: ESCC_3_177.xlsx
Input columns: ['filename', 'ESCC_1', 'ESCC_2', 'ESCC_3']
Number of cases: 177

=== Fleiss' kappa ===
Fleiss' kappa: 0.666
P_bar (mean agreement): 0.787
P_e (chance agreement): 0.363

=== Pairwise Cohen's kappa ===
ESCC_1 vs ESCC_2: Unweighted = 0.751, Quadratic-weighted = 0.889
ESCC_1 vs ESCC_3: Unweighted = 0.588, Quadratic-weighted = 0.793
ESCC_2 vs ESCC_3: Unweighted = 0.665, Quadratic-weighted = 0.842

=== Consensus ESCC distribution ===
ESCC_consensus
1b    24
1c     6
2     68
3     79
Name: count, dtype: int64

Number of full-disagreement cases: 3

Saved consensus results to: ESCC_3_with_consensus_177.xlsx


In [3]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Dataset Construction for CNN-based ESCC Classification
[177-case version | submission-ready]

This script performs the following steps:
1. Detects the working base directory automatically (/content or /mnt/data).
2. Unzips the MRI image archive if needed.
3. Loads consensus ESCC labels from Excel.
4. Scans the image directory and maps numeric IDs to real image filenames.
5. Matches each case in the Excel file to its image.
6. Generates dataset_177.csv for CNN training.

Expected input files:
    - ESCC_3_with_consensus_177.xlsx
    - images177.zip
      or extracted folder:
    - images177/

Output:
    - dataset_177.csv

Required Excel columns:
    - filename
    - ESCC_consensus
"""

import os
import zipfile
import pandas as pd


# ---------------------------------------------------------------------
# Utility functions
# ---------------------------------------------------------------------
def detect_base_dir():
    """
    Automatically detect the working base directory.

    Priority:
    1. /content   (Google Colab standard)
    2. /mnt/data  (some notebook/container environments)
    """
    candidates = ["/content", "/mnt/data"]

    for base in candidates:
        if os.path.exists(base):
            return base

    raise FileNotFoundError(
        "No valid base directory found. Neither '/content' nor '/mnt/data' exists."
    )


def resolve_existing_path(*paths):
    """
    Return the first existing path among candidates.
    If none exists, return the first candidate for default output use.
    """
    for p in paths:
        if os.path.exists(p):
            return p
    return paths[0]


def unzip_if_needed(zip_file, image_dir, extract_to):
    """
    Unzip image archive if the extracted folder does not exist.
    """
    if os.path.exists(image_dir) and os.path.isdir(image_dir):
        print(f"[INFO] Image directory already exists: {image_dir}")
        return

    if not os.path.exists(zip_file):
        raise FileNotFoundError(
            f"ZIP file not found: {zip_file}\n"
            f"Please confirm that 'images177.zip' has been uploaded."
        )

    print(f"[INFO] Unzipping: {zip_file}")
    with zipfile.ZipFile(zip_file, "r") as zf:
        zf.extractall(extract_to)
    print("[INFO] Unzip completed.")


def build_file_map(image_dir):
    """
    Build mapping from numeric file stem to actual filename.

    Example:
        '12.png' -> file_map['12'] = '12.png'
    """
    if not os.path.exists(image_dir):
        raise FileNotFoundError(f"Image directory not found: {image_dir}")

    img_files = os.listdir(image_dir)
    file_map = {}

    for f in img_files:
        full_path = os.path.join(image_dir, f)
        if not os.path.isfile(full_path):
            continue

        name, ext = os.path.splitext(f)
        if name.isdigit():
            file_map[name] = f

    return file_map


def normalize_filename_key(raw_value):
    """
    Normalize Excel filename values such as:
        12, 12.0, '12', '12.png'
    into:
        '12'
    """
    if pd.isna(raw_value):
        return None

    s = str(raw_value).strip()

    # Remove extension if present
    s_no_ext = os.path.splitext(s)[0]

    # Try numeric normalization first
    try:
        return str(int(float(s_no_ext)))
    except Exception:
        return s_no_ext.strip()


# ---------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------
def main():
    base_dir = detect_base_dir()
    print(f"[INFO] Detected base directory: {base_dir}")

    # Input/output paths
    zip_file = resolve_existing_path(
        os.path.join("/content", "images177.zip"),
        os.path.join("/mnt/data", "images177.zip"),
    )

    excel_path = resolve_existing_path(
        os.path.join("/content", "ESCC_3_with_consensus_177.xlsx"),
        os.path.join("/mnt/data", "ESCC_3_with_consensus_177.xlsx"),
    )

    image_dir = resolve_existing_path(
        os.path.join("/content", "images177"),
        os.path.join("/mnt/data", "images177"),
    )

    output_csv = os.path.join(base_dir, "dataset_177.csv")

    print("\n=== Path Configuration ===")
    print(f"ZIP_FILE   : {zip_file}")
    print(f"IMAGE_DIR  : {image_dir}")
    print(f"EXCEL_PATH : {excel_path}")
    print(f"OUTPUT_CSV : {output_csv}")

    # 1. Unzip if needed
    unzip_if_needed(zip_file, image_dir, base_dir)

    # Re-resolve image_dir in case it was created by unzip
    image_dir = resolve_existing_path(
        os.path.join("/content", "images177"),
        os.path.join("/mnt/data", "images177"),
    )

    # 2. Check image directory
    print("\n=== Directory Check ===")
    print(f"Image directory: {image_dir}")

    if not os.path.exists(image_dir):
        raise FileNotFoundError(f"Image directory not found after unzip: {image_dir}")

    image_list = os.listdir(image_dir)
    print(f"Total files in image directory: {len(image_list)}")
    print(f"First 10 files: {image_list[:10]}")

    # 3. Load consensus labels
    if not os.path.exists(excel_path):
        raise FileNotFoundError(
            f"Excel file not found: {excel_path}\n"
            f"Please confirm that 'ESCC_3_with_consensus_177.xlsx' exists."
        )

    print(f"\n[INFO] Loading label file: {excel_path}")
    df = pd.read_excel(excel_path)

    print(f"Number of rows in Excel: {len(df)}")
    print(f"Columns: {list(df.columns)}")

    required_cols = ["filename", "ESCC_consensus"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns in Excel: {missing_cols}")

    # 4. Build image filename mapping
    file_map = build_file_map(image_dir)
    print(f"Mapped numeric image IDs: {len(file_map)}")

    # 5. Match Excel rows to images
    rows = []
    missing = []

    for _, row in df.iterrows():
        key = normalize_filename_key(row["filename"])

        if key is None:
            missing.append(row["filename"])
            continue

        if key in file_map:
            rows.append({
                "filename": file_map[key],
                "label": str(row["ESCC_consensus"]).strip()
            })
        else:
            missing.append(key)

    # 6. Save dataset
    dataset = pd.DataFrame(rows)

    if dataset.empty:
        raise ValueError(
            "No images were matched. Please check whether the Excel 'filename' column "
            "corresponds to the numeric image file names in the extracted folder."
        )

    dataset.to_csv(output_csv, index=False)

    # 7. Summary
    print("\n=== Dataset Summary ===")
    print(f"Total mapped images: {len(dataset)}")

    if len(missing) > 0:
        print(f"Missing files: {len(missing)}")
        print(f"Example missing IDs: {missing[:10]}")
    else:
        print("All images successfully mapped.")

    print(f"\nSaved dataset file: {output_csv}")

    print("\n=== Label Distribution ===")
    print(dataset["label"].value_counts(dropna=False).sort_index())

    print("\n[INFO] Dataset construction completed successfully.")


if __name__ == "__main__":
    main()


[INFO] Detected base directory: /content

=== Path Configuration ===
ZIP_FILE   : /content/images177.zip
IMAGE_DIR  : /content/images177
EXCEL_PATH : /content/ESCC_3_with_consensus_177.xlsx
OUTPUT_CSV : /content/dataset_177.csv
[INFO] Unzipping: /content/images177.zip
[INFO] Unzip completed.

=== Directory Check ===
Image directory: /content/images177
Total files in image directory: 177
First 10 files: ['147.PNG', '161.PNG', '64.PNG', '52.PNG', '139.png', '165.PNG', '63.PNG', '31.png', '206.PNG', '164.PNG']

[INFO] Loading label file: /content/ESCC_3_with_consensus_177.xlsx
Number of rows in Excel: 177
Columns: ['filename', 'ESCC_1', 'ESCC_2', 'ESCC_3', 'ESCC_consensus']
Mapped numeric image IDs: 177

=== Dataset Summary ===
Total mapped images: 177
All images successfully mapped.

Saved dataset file: /content/dataset_177.csv

=== Label Distribution ===
label
1b    24
1c     6
2     68
3     79
Name: count, dtype: int64

[INFO] Dataset construction completed successfully.


In [5]:
from google.colab import drive
import glob

drive.mount("/content/drive")

scripts = glob.glob("/content/JBHI_nested_leakage_free_pipeline*.py")
print(scripts)

script = scripts[0]

!python "$script" \
  --base-dir /content \
  --output-dir /content/drive/MyDrive/JBHI_NESTED_RESULTS_CORRECTED

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['/content/JBHI_nested_leakage_free_pipeline.py']
Base directory: /content
Output directory: /content/drive/MyDrive/JBHI_NESTED_RESULTS_CORRECTED
Validated 177 explicitly linked cases.
Operation dates: 2009-11-13 to 2021-04-16
Expert_label  1b  1c   2   3
outer_fold                  
1              5   1  14  16
2              5   2  14  15
3              5   1  13  16
4              5   1  13  16
5              4   1  14  16
CNN device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 248MB/s]
/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/shap/explaine

In [6]:
!cd /content/drive/MyDrive && zip -r JBHI_results_corrected_for_review.zip JBHI_NESTED_RESULTS_CORRECTED

  adding: JBHI_NESTED_RESULTS_CORRECTED/ (stored 0%)
  adding: JBHI_NESTED_RESULTS_CORRECTED/validated_case_linkage_and_folds.csv (deflated 65%)
  adding: JBHI_NESTED_RESULTS_CORRECTED/fold_distribution.csv (deflated 41%)
  adding: JBHI_NESTED_RESULTS_CORRECTED/input_validation_manifest.json (deflated 39%)
  adding: JBHI_NESTED_RESULTS_CORRECTED/cnn_cache/ (stored 0%)
  adding: JBHI_NESTED_RESULTS_CORRECTED/cnn_cache/primary_exclude_1_predictions.csv (deflated 50%)
  adding: JBHI_NESTED_RESULTS_CORRECTED/cnn_cache/primary_exclude_1_metadata.json (deflated 38%)
  adding: JBHI_NESTED_RESULTS_CORRECTED/cnn_cache/primary_exclude_2_predictions.csv (deflated 49%)
  adding: JBHI_NESTED_RESULTS_CORRECTED/cnn_cache/primary_exclude_2_metadata.json (deflated 38%)
  adding: JBHI_NESTED_RESULTS_CORRECTED/cnn_cache/primary_exclude_3_predictions.csv (deflated 49%)
  adding: JBHI_NESTED_RESULTS_CORRECTED/cnn_cache/primary_exclude_3_metadata.json (deflated 37%)
  adding: JBHI_NESTED_RESULTS_CORRECTED/c

In [7]:
!cd /content/drive/MyDrive/JBHI_NESTED_RESULTS_CORRECTED && zip -r ../JBHI_results_corrected_for_review_small.zip \
  tables figures completed_run_manifest.json fold_distribution.csv input_validation_manifest.json

  adding: tables/ (stored 0%)
  adding: tables/case_level_primary_nested_predictions.xlsx (deflated 5%)
  adding: tables/case_level_temporal_predictions.xlsx (deflated 5%)
  adding: tables/JBHI_nested_results.xlsx (deflated 1%)
  adding: figures/ (stored 0%)
  adding: figures/oof_shap_heatmap.png (deflated 27%)
  adding: figures/primary_3M_roc_calibration.png (deflated 18%)
  adding: figures/primary_6M_roc_calibration.png (deflated 18%)
  adding: figures/primary_12M_roc_calibration.png (deflated 19%)
  adding: figures/temporal_3M_roc_calibration.png (deflated 22%)
  adding: figures/temporal_6M_roc_calibration.png (deflated 21%)
  adding: figures/temporal_12M_roc_calibration.png (deflated 21%)
  adding: completed_run_manifest.json (deflated 40%)
  adding: fold_distribution.csv (deflated 41%)
  adding: input_validation_manifest.json (deflated 39%)


In [8]:
from google.colab import files
files.download("/content/drive/MyDrive/JBHI_results_corrected_for_review_small.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>